# Summary_Day16_offline.ipynb  
## CNN 스크래치 구현 · 인터넷 불가 버전 · FakeData / Synthetic SVHN

이 파일은 **인터넷이 안 되는 환경**에서 16강 CNN 스크래치 구현 흐름을 연습하기 위한 버전이다.

원본 강의는 CIFAR-10과 SVHN을 다운로드해서 사용한다.  
인터넷이 없으면 다운로드가 실패할 수 있다.

그래서 offline 버전은 다음 방식으로 구성한다.

```text
CIFAR-10 대신 FakeData 사용
SVHN 대신 직접 만든 숫자 패턴 이미지 사용
CNN 구조는 동일하게 연습
Conv2d / Pool / Flatten / Linear / Hook / Residual Block 흐름 유지
```

> 주의:  
> offline 버전은 진짜 CIFAR-10/SVHN 성능을 보는 파일이 아니다.  
> 인터넷 없이 CNN 구조와 코드 흐름을 실행하기 위한 대체 실습이다.

## 1. 라이브러리 준비

다운로드 없이 실행하기 위해 `FakeData`와 직접 만든 Tensor 데이터를 사용한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import random
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)

## 2. FakeData로 CIFAR-10식 데이터 만들기

`FakeData`는 인터넷 없이 가짜 이미지와 label을 만들어 준다.

### 함수 사용법

```python
datasets.FakeData(size=1000, image_size=(3,32,32), num_classes=10, transform=transform)
```

- `size`: 샘플 수다.
- `image_size`: 이미지 shape이다.
- `num_classes`: class 개수다.
- 다운로드가 필요 없다.

In [ ]:
fake_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

fake_train = datasets.FakeData(
    size=1000,
    image_size=(3, 32, 32),
    num_classes=10,
    transform=fake_transform
)

fake_test = datasets.FakeData(
    size=300,
    image_size=(3, 32, 32),
    num_classes=10,
    transform=fake_transform
)

train_loader = DataLoader(fake_train, batch_size=64, shuffle=True)
test_loader = DataLoader(fake_test, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))

print("images:", images.shape)
print("labels:", labels.shape)

## 3. SimpleCNN 모델 구현

온라인 버전의 CIFAR-10 SimpleCNN과 같은 구조다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=10).to(device)

dummy = torch.randn(1, 3, 32, 32).to(device)

with torch.no_grad():
    feature_out = model.features(dummy)
    output = model(dummy)

print(model)
print("feature_out:", feature_out.shape)
print("output:", output.shape)

## 4. 학습/평가 함수

가짜 데이터라도 학습 루프 구조는 동일하다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        pred = out.argmax(dim=1)

        total_loss += loss.item() * y.size(0)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / total, 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            loss = criterion(out, y)

            pred = out.argmax(dim=1)

            total_loss += loss.item() * y.size(0)
            correct += (pred == y).sum().item()
            total += y.size(0)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    return total_loss / total, 100.0 * correct / total, np.array(y_true), np.array(y_pred)

## 5. FakeData 짧은 학습 실행

FakeData는 랜덤 데이터라 성능 자체는 큰 의미가 없다.  
여기서는 CNN 학습 코드가 끝까지 실행되는지 확인한다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = []

for epoch in range(2):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    history.append([epoch + 1, train_loss, train_acc, test_loss, test_acc])

    print(
        f"epoch {epoch + 1} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.2f}% | "
        f"test_loss={test_loss:.4f} | test_acc={test_acc:.2f}%"
    )

history = np.array(history)

In [ ]:
plt.plot(history[:, 0], history[:, 1], label="train loss")
plt.plot(history[:, 0], history[:, 3], label="test loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Offline SimpleCNN Loss")
plt.legend()
plt.show()

plt.plot(history[:, 0], history[:, 2], label="train acc")
plt.plot(history[:, 0], history[:, 4], label="test acc")
plt.xlabel("epoch")
plt.ylabel("accuracy (%)")
plt.title("Offline SimpleCNN Accuracy")
plt.legend()
plt.show()

## 6. 직접 만든 숫자 패턴 데이터

SVHN 대신 간단한 숫자 패턴 이미지를 직접 만든다.  
각 class마다 밝은 줄 위치가 달라지도록 만들어 10 class 분류 구조를 연습한다.

In [ ]:
def make_synthetic_digits(n_per_class=80, image_size=32):
    images = []
    labels = []

    for digit in range(10):
        for i in range(n_per_class):
            img = np.random.normal(0, 0.08, size=(3, image_size, image_size)).astype(np.float32)

            row = 2 + (digit * 3) % 24
            col = 2 + (digit * 5) % 24
            ch = digit % 3

            img[ch, row:row + 5, :] += 0.9
            img[ch, :, col:col + 3] += 0.7

            if digit % 2 == 0:
                for k in range(6, 26):
                    img[ch, k, k] += 0.6
            else:
                for k in range(6, 26):
                    img[ch, k, 31 - k] += 0.6

            img = np.clip(img, -1, 1)

            images.append(img)
            labels.append(digit)

    X = np.stack(images)
    y = np.array(labels)

    idx = np.random.permutation(len(y))

    return torch.FloatTensor(X[idx]), torch.LongTensor(y[idx])

X, y = make_synthetic_digits(n_per_class=80)

n_train = int(len(y) * 0.8)

X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

digit_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
digit_test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(20):
    plt.subplot(2, 10, i + 1)

    img = X_train[i].permute(1, 2, 0).numpy()
    img = (img + 1) / 2
    img = np.clip(img, 0, 1)

    plt.imshow(img)
    plt.title(str(y_train[i].item()), fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 7. SVHN_CNN 구조를 synthetic digits에 적용

온라인 버전의 SVHN_CNN 구조를 그대로 사용한다.

In [ ]:
class SVHN_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

digit_model = SVHN_CNN(num_classes=10).to(device)

print(digit_model)

## 8. hook으로 shape 추적

중간 layer의 출력 shape을 확인한다.

In [ ]:
def trace_layer_shapes(model, input_shape=(1, 3, 32, 32), device="cpu"):
    layer_outputs = OrderedDict()
    handles = []

    def hook_fn(module, input, output):
        layer_name = module.__class__.__name__
        count = sum(1 for key in layer_outputs.keys() if layer_name in key)

        if count > 0:
            layer_name = f"{layer_name}_{count + 1}"

        if isinstance(output, torch.Tensor):
            layer_outputs[layer_name] = tuple(output.shape)

    for name, module in model.named_modules():
        if len(list(module.children())) == 0 and module != model:
            handles.append(module.register_forward_hook(hook_fn))

    dummy = torch.randn(*input_shape).to(device)

    model.eval()

    with torch.no_grad():
        _ = model(dummy)

    for handle in handles:
        handle.remove()

    return layer_outputs

shape_trace = trace_layer_shapes(digit_model, device=device)

for name, shape in shape_trace.items():
    print(f"{name:<20} {shape}")

## 9. synthetic digits 학습 실행

직접 만든 데이터라 인터넷 없이 CNN 학습 흐름을 연습할 수 있다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(digit_model.parameters(), lr=0.001, weight_decay=1e-4)

digit_history = []

for epoch in range(3):
    train_loss, train_acc = train_one_epoch(digit_model, digit_train_loader, criterion, optimizer, device)
    test_loss, test_acc, y_true, y_pred = evaluate(digit_model, digit_test_loader, criterion, device)

    digit_history.append([epoch + 1, train_loss, train_acc, test_loss, test_acc])

    print(
        f"epoch {epoch + 1} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.2f}% | "
        f"test_loss={test_loss:.4f} | test_acc={test_acc:.2f}%"
    )

digit_history = np.array(digit_history)

## 10. Residual Block 구조 연습

인터넷 없이도 Residual Block 구조는 그대로 연습할 수 있다.

In [ ]:
class BasicResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = out + identity
        out = F.relu(out)

        return out

block = BasicResidualBlock(32, 64, stride=2).to(device)

x = torch.randn(2, 32, 32, 32).to(device)

with torch.no_grad():
    y = block(x)

print("입력:", x.shape)
print("출력:", y.shape)

## 11. Confusion Matrix 확인

synthetic digits 기준으로 class별 오류를 확인한다.

In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate(digit_model, digit_test_loader, criterion, device)

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("Synthetic Digits Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(10), [str(i) for i in range(10)])
plt.yticks(range(10), [str(i) for i in range(10)])

for (i, j), value in np.ndenumerate(cm):
    if value > 0:
        plt.text(j, i, str(value), ha="center", va="center", fontsize=8)

plt.colorbar()
plt.show()

print(classification_report(y_true, y_pred, target_names=[str(i) for i in range(10)], zero_division=0))

## 12. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `FakeData` | 가짜 이미지 데이터셋 | 인터넷 없이 구조 연습 |
| `TensorDataset` | Tensor를 Dataset으로 묶음 | 직접 만든 데이터에 사용 |
| `SimpleCNN` | 기본 CNN 모델 | CIFAR-10식 구조 |
| `SVHN_CNN` | 숫자 이미지 CNN | 3 conv block + classifier |
| `hook` | 중간 출력 추적 | `register_forward_hook` |
| `Residual Block` | skip connection block | `F(x)+x` |
| `shortcut` | 건너뛰는 연결 | shape 맞춤 |
| `AdamW` | Adam + weight decay | 과적합 완화 |
| `weight_decay` | L2 정규화 계열 | 파라미터 크기 규제 |

## 13. 시험용 요약

```text
오프라인 16강 핵심 = 다운로드 없이 CNN 구조와 학습 루프를 실행한다
```

꼭 기억할 것:

- 원본 강의는 CIFAR-10과 SVHN 다운로드를 사용한다.
- 인터넷이 없으면 FakeData나 직접 만든 Tensor 데이터로 구조를 연습할 수 있다.
- `Conv2d → ReLU → MaxPool2d`는 CNN의 기본 block이다.
- `Flatten → Linear`는 classifier 쪽 구조다.
- hook은 중간 layer 출력 shape을 추적한다.
- Residual Block은 입력을 출력에 더하는 skip connection 구조다.
- shape이 다르면 shortcut에 1×1 Conv를 사용한다.